In [ ]:
# load librairies
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import os
from os.path import join
import seaborn as sns
import statsmodels.api as sm
import scipy
import json
import statsmodels.formula.api as smf
from collections import defaultdict
from scipy.stats import wilcoxon, ttest_rel

from functions import utils, plotting, io, behav_analysis

In [ ]:
# pick format to save figures : png for quick visualization, pdf for illustrator
save_as = 'png'
color_dict = {
    'DBS ON': '#20a39e', 
    'DBS OFF': '#ef5b5b', 
    'control': '#ffba49', 
    'preop': '#8E7DBE',
    'Session 1': "#206ea1", 
    'Session 2': "#5FA363", 
    }
visualize_by = 'subject' # pick whether to visualize by condition (DBS ON, DBS OFF, control) or by subject id


# 1. Select subjects, create color palette and load data #

In [ ]:
# Use the Included or excluded.xlsx document:
onedrive_path = utils._get_onedrive_path()

working_path = os.path.dirname(os.getcwd())
results_path = join(working_path, "results")
behav_results_saving_path = join(results_path, "behav_results")
if not os.path.isdir(behav_results_saving_path):
    os.makedirs(behav_results_saving_path)

# read the json file containing the included and excluded subjects
# Open and read the JSON file
included_excluded_file = join(results_path, 'final_included_subjects.json')
with open(included_excluded_file, 'r') as file:
    included_subjects = json.load(file)    

In [ ]:
included_subjects

In [ ]:
subject_colors = utils.create_color_palette(included_subjects)
utils.plot_color_palette(
    subject_colors = subject_colors,
    save_as='png', 
    saving_path=behav_results_saving_path
    )

In [ ]:
# load excel files for all included subjects and extract stats
data = io.load_behav_data(included_subjects, onedrive_path)
stats = utils.extract_stats(data)

# 2. Get scales from 'WP3_rec_info.xlsx' and convert to panda dataframe containing all included subjects measures, save to excel #

In [ ]:
# fetch excel file containing recording and subject information
excel_file = join(onedrive_path, 'WP3_rec_info.xlsx')
# select the relevant sheet called 'Subjects List'
subject_info_df = pd.read_excel(excel_file, sheet_name='Subjects List')
control_info_df = pd.read_excel(excel_file, sheet_name='Controls List')

# get session order and save it to 'session_info.txt'
behav_analysis.get_session_order(
    included_subjects = included_subjects,
    subject_info_df = subject_info_df,
    behav_results_saving_path = behav_results_saving_path
)


In [ ]:
# get the behavioral scores for all scales of interest
scale_names = ['UPDRS_ON', 'UPDRS_OFF', 'MOCA', 'BDI', 'SAS', 'BIS_TOTAL', 'BIS_nonplanning', 'BIS_motor', 'BIS_attentional', 'OCI_TOTAL', 'OCI_washing', 'OCI_obsessing', 'OCI_hoarding', 'OCI_ordering', 'OCI_checking', 'OCI_neutralizing']
subs = [subj.split(' ')[0] for subj in included_subjects]
# remove duplicates in subs and order the list:
subs = sorted(list(set(subs)))


sub_scale_dict = {}
for sub in subs:
    if sub.startswith('C'):
        sub_line = control_info_df[control_info_df['StudyCode'] == sub]
    else:
        sub_line = subject_info_df[subject_info_df['StudyCode'] == sub]
    scale_dict = {}
    for scale in scale_names:
        if scale in sub_line.columns:
            scale_value = sub_line[scale].values[0]
        else:
            scale_value = None
        scale_dict[scale] = scale_value
    sub_scale_dict[sub] = scale_dict    

In [ ]:
sub_scale_dict

In [ ]:
# save sub_scale_dict to an excel file:
sub_scale_df = pd.DataFrame.from_dict(sub_scale_dict, orient='index')
sub_scale_df.to_excel(join(behav_results_saving_path, 'sub_scale_dict.xlsx'))


In [ ]:
sub_scale_df

# 3. Visualize results from scales #

## 3.1. Visualize score per subject for each scale ##

In [ ]:
temp_save = join(behav_results_saving_path, "scales")
if not os.path.isdir(temp_save):
    os.makedirs(temp_save)

for viz in ['subject', 'condition']:
    behav_analysis.visualize_scales_scores(
        scale_names = scale_names,
        sub_scale_dict = sub_scale_dict,
        subs = subs,
        subject_colors = subject_colors,
    color_dict = color_dict,
    visualize_by = viz,
    saving_path = temp_save,
    save_as = save_as,
    show_plot = True
    )

In [ ]:
temp_save = join(behav_results_saving_path, "scales")
if not os.path.isdir(temp_save):
    os.makedirs(temp_save)

for viz in ['subject', 'condition']:
    behav_analysis.visualize_updrs_scores(
    sub_scale_dict = sub_scale_dict, 
    subject_colors = subject_colors,
    color_dict = color_dict, 
    colored_by = viz,
    saving_path = temp_save, 
    save_as = save_as,
    show_plot = True
    )

## 3.2. Correlate scores between 2 scales and visualize ##

In [ ]:
temp_save = join(behav_results_saving_path, "scales")
if not os.path.isdir(temp_save):
    os.makedirs(temp_save)  
      
behav_analysis.correlate_two_scales(
    scale_1 = 'SAS',
    scale_2 = 'BDI',
    sub_scale_dict = sub_scale_dict,
    subject_colors = subject_colors,
    saving_path = temp_save,
    save_as = save_as,
    show_plot = True
)

behav_analysis.correlate_two_scales(
    scale_1 = 'OCI_TOTAL',
    scale_2 = 'BIS_TOTAL',
    sub_scale_dict = sub_scale_dict,
    subject_colors = subject_colors,
    saving_path = temp_save,
    save_as = save_as,
    show_plot = True
)